In [ ]:
from AI4AO import PyramidWFS, PhaseDataset, DeformableMirror, InfinitePhaseScreen
from AI4AO import imshow, imshow_multiple
from kalman.Kalman_tools import *
import torch
device = "cuda"
from dataclasses import dataclass
import pylab as plt

In [ ]:
@dataclass
class EvaluationResult:
    """Trajectories from a closed-loop evaluation rollout, each shaped (n_steps, ...)."""
    opd: torch.Tensor
    pupil: torch.Tensor
    opd_reconstructed: torch.Tensor
    residual_opd: torch.Tensor
    wfs_frames: torch.Tensor
    psfs: torch.Tensor
    z_output: torch.Tensor

In [ ]:
WFSParams = dict(
    {
        "Nres":30,
        "sampling":90/30,
        "D": 1.5,
        "centralObstruction": 0.,
        "useNoise": True,
        "Modulation": 2,
        "Wavelength": 635e-9,
        "Nphotons": [4.5, 6],  # Log range of number of photons in measurement
        "RON": [1, 3],  # Read-out noise in photons per pixel per frame
    }
)


## Atmosphere parameters
AtmosParams = dict(
    {
        "r0": [0.07, 0.07],  # Fired parameter range (m)
        "L0": [10, 30.0000],  # Outter scale range (m)
        "Nphases": 1,  # Number of phases in the batch
        "Layers": [5, 10],  # Number of layers in phase range
        "f_slope": 11.0 / 6.0,  # Slope of the spectrum (11/6) is the default
        "Scintillation": False # Use or not scinitillation
    }
)


## Loop parameters
LoopParams = dict(
    {
        "loopFrequency": 500,
        "delayFrames": 2,
        "windSpeedVector": [1, 10],
        "levelOfCorrection": [0.9, 1.],
        "loopGain": [0.2, 0.5],
        "loopLeak": [0.9,1.] 
    }
)

## Atmosphere parameters
DMParams = dict(
    {
        "Nactuator": 11,
        "Nmodes": 80,
        "FlipLeftRight": False,
        "FlipTopBottom": False,
        "moffatParam": 2,
        "signedAmplitude": 1e-5,
        "MechCoupling": 0.35
    }
)

In [ ]:
dataset = PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)
# dataset.subharmonicLevels=3

In [ ]:
wfs = PyramidWFS(WFSParams, device)
wfs.eval()
dm = DeformableMirror(WFSParams, DMParams, device)
dm.eval()
M2C = dm.MakeZernikeM2C()*10

modes = dm(M2C.T)
wfs.BuildReconstructionMatrix(modes)

In [ ]:
from collections import deque

M2C_T = M2C.T
Nmodes = M2C.shape[1]

# Loop delay: the command computed at frame k first appears in the WFS measurement z_{k+delay}.
# The WFS sees the residual of the previous frame (1 frame) and the DM applies dm_buffer[0],
# the command from delay - 1 frames ago. delay = 2 is the old z_buffer1/z_buffer2 chain,
# delay = 3 adds one buffer.
delay = 1

batch = dataset[0]
opd_gt = batch["opd"]
pupilGT = batch["pupil"]
gain = 0.3
leak = 0.995
photons = batch["nphotons"]
ron = batch["ron"]

wfs.SetPhotonsAndRON(photons, ron)

z_estimated = torch.zeros(opd_gt.shape[0], Nmodes, device=device)
dm_buffer = deque([torch.zeros_like(z_estimated)] * delay, maxlen=delay)   # [u_{k-delay+1}, ..., u_k]
z_output = torch.zeros_like(z_estimated)
opd_reconstructed = torch.zeros_like(opd_gt)
residual_opd = torch.zeros_like(opd_gt)

opds, pupils, reconstructed, residuals, frames, psfs, outputs = [], [], [], [], [], [], []
wfs_measurements, dm_commands = [], []

n_steps = 5000

for i in range(n_steps):
    if i > 0:
        batch = dataset[i]
        opd_gt = batch["opd"]
        pupilGT = batch["pupil"]

    
    wfs_frames = wfs(residual_opd, pupilGT)
    psf = wfs.GetPSF(residual_opd, pupilGT)
    z_output = wfs.GetReconstructedOPD(wfs_frames)

    z_estimated = z_estimated * leak + gain * z_output

    dm_buffer.append(z_estimated)
    opd_reconstructed = dm(dm_buffer[0] @ M2C_T)
    residual_opd = opd_gt - opd_reconstructed


    opds.append(opd_gt)
    pupils.append(pupilGT)
    reconstructed.append(opd_reconstructed)
    residuals.append(residual_opd)
    frames.append(wfs_frames)
    psfs.append(psf)
    outputs.append(z_output)

    if i > n_steps * 0.1:
        wfs_measurements.append(z_output)
        dm_commands.append(z_estimated)

result =  EvaluationResult(
            opd = torch.stack(opds),
            pupil = torch.stack(pupils),
            opd_reconstructed = torch.stack(reconstructed),
            residual_opd = torch.stack(residuals),
            wfs_frames = torch.stack(frames),
            psfs = torch.stack(psfs),
            z_output = torch.stack(outputs),
        )

kalmanData = KalmanData(
    wfs_measurements=torch.stack(wfs_measurements).squeeze(),
    modal_dm_commands=torch.stack(dm_commands).squeeze()
)

In [ ]:
x = estimate_atmospheric_state(kalmanData.wfs_measurements, kalmanData.modal_dm_commands, delay)
A = estimate_A(x)
R = estimate_R(wfs.reconstructionMatrix, wfs.reference_intensity, wfs.Nphotons, wfs.RON)
Q = estimate_raw_Q(x, A)
# Q = estimate_Q(Q, A, R)
H = torch.eye(dm.nModes, device=device)
P0 = R

In [ ]:
matrices = KalmanMatrices(
    A=A,
    H=H,
    Q=Q,
    R=R,
)

state = KalmanState(
    x=x[-1],
    P=P0,
)

In [ ]:
imshow(A, colorbar=True)

In [ ]:
residual_opd_kalman = torch.clone(residual_opd)
dm_buffer_kalman = deque([torch.clone(u) for u in dm_buffer], maxlen=delay)
A_delay = torch.linalg.matrix_power(A, delay)

opds, pupils, reconstructed, residuals, frames, psfs, outputs = [], [], [], [], [], [], []

for i in range(n_steps, n_steps+2000):
    if i > 0:
        batch = dataset[i]
        opd_gt = batch["opd"]
        pupilGT = batch["pupil"]

    wfs_frames = wfs(residual_opd_kalman, pupilGT)
    psf = wfs.GetPSF(residual_opd_kalman, pupilGT)
    z_output = wfs.GetReconstructedOPD(wfs_frames)

    # Command that was on the DM during this measurement: u_{k-delay}
    u_dm = dm_buffer_kalman[0]

    # Convert residual measurement into an atmospheric measurement,
    # x_k = z_k + u_{k-delay}, the same as estimate_atmospheric_state
    x_measurement = z_output + u_dm

    # Estimate atmosphere
    state, K = kalman_filter(
        measurement=x_measurement.squeeze(),
        state=state,
        matrices=matrices,
    )

    # The command computed now is first seen in z_{k+delay}: predict delay steps ahead (no integrator)
    z_estimated_kalman = (A_delay @ state.x).view(1, -1)

    dm_buffer_kalman.append(z_estimated_kalman)
    opd_reconstructed = dm(dm_buffer_kalman[0] @ M2C_T)
    residual_opd_kalman = opd_gt - opd_reconstructed


    opds.append(opd_gt)
    pupils.append(pupilGT)
    reconstructed.append(opd_reconstructed)
    residuals.append(residual_opd_kalman)
    frames.append(wfs_frames)
    psfs.append(psf)
    outputs.append(z_output)

result_kalman =  EvaluationResult(
            opd = torch.stack(opds),
            pupil = torch.stack(pupils),
            opd_reconstructed = torch.stack(reconstructed),
            residual_opd = torch.stack(residuals),
            wfs_frames = torch.stack(frames),
            psfs = torch.stack(psfs),
            z_output = torch.stack(outputs),
        )


In [ ]:
imshow(state.P - P0, colorbar=True)

In [ ]:
import matplotlib as mpl
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import pylab as plt

result_to_show = result_kalman

fig, axes = imshow_multiple(
    [
        {"tensor": result_to_show.opd[0], "title": "Input OPD", "same_scale": True},
        {"tensor": result_to_show.residual_opd[0], "title": "Residual OPD", "scale_reference": result_to_show.opd[0]},
        {"tensor": result_to_show.wfs_frames[0], "title": "WFS frame"},
        {"tensor": torch.sqrt(result_to_show.psfs[0]), "title": "PSF", "same_scale": True},
    ],
    max_channel_number = 9
)


def update(i):
    imshow_multiple(
        [
            {"tensor": result_to_show.opd[i], "title": "Input OPD", "same_scale": True},
            {"tensor": result_to_show.residual_opd[i], "title": "Residual OPD", "scale_reference": result_to_show.opd[i]},
            {"tensor": result_to_show.wfs_frames[i], "title": "WFS frame"},
            {"tensor": torch.sqrt(result_to_show.psfs[i]), "title": "PSF", "same_scale": True},
        ],
        fig=fig, axes=axes, 
        max_channel_number = 9
    )
    return [ax.images[0] for tensor_axes in axes for ax in tensor_axes]


anim = FuncAnimation(fig, update, frames=50, interval=50, blit=False)
plt.close(fig)

mpl.rcParams["animation.embed_limit"] = 200
HTML(anim.to_jshtml())


In [ ]:
plt.plot(torch.arange(n_steps),100* torch.exp(-(2* torch.pi / dataset.wavelength * result.residual_opd).var(dim=(-2,-1)).cpu()))
plt.plot(torch.arange(n_steps, n_steps+2000), 100* torch.exp(-(2* torch.pi / dataset.wavelength * result_kalman.residual_opd).var(dim=(-2,-1)).cpu()))
plt.legend(['Classic integrator', 'Kalman Filter'])
plt.ylabel('Strehl ratio (%)')
plt.xlabel('Iteration number')
plt.show()

In [ ]:
plt.plot(result.psfs.mean(dim = 0).squeeze()[40:50,45].cpu())
plt.plot(result_kalman.psfs.mean(dim = 0).squeeze()[40:50,45].cpu())

In [ ]:
imshow(torch.log(torch.stack((result.psfs.mean(dim = 0).squeeze(), result_kalman.psfs.mean(dim = 0).squeeze()))))